## Mercedes Task 2 - New Sign-Convention Pipeline

In [1]:
import numpy as np

# ============================================================
# Task 2 (New Pipeline): Triangulation with explicit Y-sign convention
# ============================================================
# We keep the provided calibration matrices and transform.
# We explicitly map image v (downward) to sensor Y-up via sign flip.

Kc = np.array([
    [4000,    0, 1924],
    [   0, 4000, 1084],
    [   0,    0,    1]
], dtype=float)

Kp = np.array([
    [750,   0, 160],
    [  0, 750,  40],
    [  0,   0,   1]
], dtype=float)

T_C_from_P = np.array([
    [0.9956, -0.0888, -0.0301,  0.50],
    [0.0871,  0.9946, -0.0551, -0.10],
    [0.0349,  0.0523,  0.9980,  0.20],
    [0.0,     0.0,     0.0,     1.0 ]
], dtype=float)

R = T_C_from_P[:3, :3]
t = T_C_from_P[:3, 3]

# Fictitious pairs from assignment
pairs = [
    ((50.00, 20.00), (1968.20, 1072.01)),
    ((250.00, 20.00), (2140.40, 1072.01)),
    ((150.00, 60.00), (2054.30, 1107.01)),
]

# Modified sign convention (Y-up in sensor frame)
CAM_Y_UP = True
PROJ_Y_UP = True


def pixel_to_ray(K, u, v, y_up=False):
    """
    Pixel -> unit ray in local sensor frame.
    x = (u-cx)/fx
    y = (v-cy)/fy, and flip if y_up=True to represent +Y upward.
    """
    fx = K[0, 0]
    fy = K[1, 1]
    cx = K[0, 2]
    cy = K[1, 2]

    x = (u - cx) / fx
    y = (v - cy) / fy
    if y_up:
        y = -y

    d = np.array([x, y, 1.0], dtype=float)
    d = d / np.linalg.norm(d)
    return d


def triangulate_midpoint(O1, d1, O2, d2):
    A = np.column_stack((d1, -d2))
    b = O2 - O1
    sol, *_ = np.linalg.lstsq(A, b, rcond=None)
    s, l = sol

    P1 = O1 + s * d1
    P2 = O2 + l * d2
    X = 0.5 * (P1 + P2)
    gap = np.linalg.norm(P1 - P2)

    return X, P1, P2, gap, s, l


# Triangulation in camera frame
Oc = np.zeros(3)
Op = t.copy()

X_list = []
gaps = []

print('=== New Pipeline: Fictitious points with modified sign convention ===')
print(f'Camera y_up={CAM_Y_UP}, Projector y_up={PROJ_Y_UP}')
print()

for i, ((up, vp), (uc, vc)) in enumerate(pairs, start=1):
    dc = pixel_to_ray(Kc, uc, vc, y_up=CAM_Y_UP)
    dp_local = pixel_to_ray(Kp, up, vp, y_up=PROJ_Y_UP)
    dp = R @ dp_local
    dp = dp / np.linalg.norm(dp)

    X, P1, P2, gap, s, l = triangulate_midpoint(Oc, dc, Op, dp)
    X_list.append(X)
    gaps.append(gap)

    print(f'Point {i}:')
    print(f'  P pixel (u_p,v_p) = ({up:.2f}, {vp:.2f})')
    print(f'  C pixel (u_c,v_c) = ({uc:.2f}, {vc:.2f})')
    print(f'  s={s:.4f}, l={l:.4f}, gap={gap:.6f} m')
    print(f'  X{i}={X}')
    print(f'  Z{i}={X[2]:.4f} m')
    print()

X1, X2, X3 = X_list

print('=== Aggregate checks ===')
print(f's>=0 count: {sum(triangulate_midpoint(Oc, pixel_to_ray(Kc, uc, vc, CAM_Y_UP), Op, (R @ pixel_to_ray(Kp, up, vp, PROJ_Y_UP)) / np.linalg.norm(R @ pixel_to_ray(Kp, up, vp, PROJ_Y_UP)))[4] >= 0 for (up,vp),(uc,vc) in pairs)}/3')
print(f'l>=0 count: {sum(triangulate_midpoint(Oc, pixel_to_ray(Kc, uc, vc, CAM_Y_UP), Op, (R @ pixel_to_ray(Kp, up, vp, PROJ_Y_UP)) / np.linalg.norm(R @ pixel_to_ray(Kp, up, vp, PROJ_Y_UP)))[5] >= 0 for (up,vp),(uc,vc) in pairs)}/3')
print(f'Z>0 count : {sum(np.array([x[2] for x in X_list]) > 0)}/3')
print(f'gap min/mean/max [m]: {np.min(gaps):.6f} / {np.mean(gaps):.6f} / {np.max(gaps):.6f}')
print()

# Plane fit
v1 = X2 - X1
v2 = X3 - X1
n = np.cross(v1, v2)

norm_n = np.linalg.norm(n)
if norm_n < 1e-12:
    raise RuntimeError('Degenerate plane fit: points are collinear or nearly identical.')

n = n / norm_n
D = -np.dot(n, X1)

print('=== Plane fit (camera frame) ===')
print('n =', n)
print(f'{n[0]:.6f} * X + {n[1]:.6f} * Y + {n[2]:.6f} * Z + {D:.6f} = 0')
print()

print('=== Residual check ===')
for i, X in enumerate(X_list, start=1):
    resid = float(np.dot(n, X) + D)
    print(f'Point {i} residual = {resid:.6e}')

print()
print('Done.')


=== New Pipeline: Fictitious points with modified sign convention ===
Camera y_up=True, Projector y_up=True

Point 1:
  P pixel (u_p,v_p) = (50.00, 20.00)
  C pixel (u_c,v_c) = (1968.20, 1072.01)
  s=2.5596, l=2.3992, gap=0.211361 m
  X1=[ 0.05235747 -0.09522955  2.5594368 ]
  Z1=2.5594 m

Point 2:
  P pixel (u_p,v_p) = (250.00, 20.00)
  C pixel (u_c,v_c) = (2140.40, 1072.01)
  s=-11.8005, l=-12.0359, gap=0.180876 m
  X2=[ -0.5884177    0.04060404 -11.78608321]
  Z2=-11.7861 m

Point 3:
  P pixel (u_p,v_p) = (150.00, 60.00)
  C pixel (u_c,v_c) = (2054.30, 1107.01)
  s=2.7279, l=2.5255, gap=0.425231 m
  X3=[ 0.24264679 -0.16233443  2.72057266]
  Z3=2.7206 m

=== Aggregate checks ===
s>=0 count: 2/3
l>=0 count: 2/3
Z>0 count : 2/3
gap min/mean/max [m]: 0.180876 / 0.272489 / 0.425231

=== Plane fit (camera frame) ===
n = [-0.33719266 -0.94141559  0.00614748]
-0.337193 * X + -0.941416 * Y + 0.006147 * Z + -0.087730 = 0

=== Residual check ===
Point 1 residual = 0.000000e+00
Point 2 residua